<a href="https://colab.research.google.com/github/NamSee04/CS114.P11/blob/main/CS114_P11_Clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# ỨNG DỤNG CLUSTERING ĐỂ PHÁT HIỆN ẢNH TRÙNG

1. Yêu cầu chung: Dùng kỹ thuật clustering để tạo công cụ hỗ trợ phát hiện các ảnh trùng nhau

2. Yêu cầu cụ thể:
  - Input: Danh sách các ảnh được lưu trong tập tin, ví dụ CarDataset-Splits-1-Train.csv (xem mô tả https://colab.research.google.com/drive/1gf0GzvW0tHddKtuvMUNIvglUT-J6oW6S?usp=sharing)
  - Output: Danh sách các clusters và hiển thị các ảnh trong cluster

3. Hướng dẫn:
  - Bước 1:
    - Mỗi ảnh cần thực hiện bước rút trích đặc trưng (feature extraction), biểu diễn dưới dạng một vector đặc trưng d chiều (d-dimension).
    - Có nhiều công cụ hỗ trợ bước rút trích đặc trưng, trong bài tập này, chúng ta sẽ chọn một công cụ sao cho tốc độ xử lý nhanh nhưng kết quả tốt. Các mô hình MobileNet (https://keras.io/api/applications/mobilenet/) có thể được dùng vì đáp ứng các tiêu chí này.
  - Bước 2:
    - Chọn một thuật toán clustering - ví dụ K-Means (số lượgn clusters K=5)
    - Ghi kết quả clustering ra tập tin - thay CategoryID bằng ClusterID
  - Bước 3:
    - Hiển thị kết quả clustering - kế thừa kết quả của bài tập Hiển thị dữ liệu https://colab.research.google.com/drive/1rHbKlJd7O9E49SsJlHnZNKcyTbwXT_Ls?usp=sharing
    - Từ kết quả hiển thị, nếu các ảnh nhìn trùng nhau, nhưng tên tập tin khác nhau thì có thể đưa vào danh sách hậu kiểm.

In [1]:
from google.colab import drive

drive.mount('/content/drive')
root_dir = '/content/drive/MyDrive/Public'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
import csv
from sklearn.model_selection import StratifiedKFold
import numpy as np
from PIL import Image



In [3]:
VALID_EXTENSIONS = ['.jpg', '.jpeg', '.png']
LABEL = {
    'Others': 0,
    'Honda': 1,
    'Hyundai': 2,
    'KIA': 3,
    'Mazda': 4,
    'Mitsubishi': 5,
    'Suzuki': 6,
    'Toyota': 7,
    'VinFast': 8,
}

def process_directory(root_dir):
    """Traverse directory and collect valid image paths and their labels."""
    data = []
    labels = []

    for car_brand in os.listdir(root_dir):
        brand_dir = os.path.join(root_dir, car_brand)
        # Skip if not a directory or brand not in LABEL
        if not os.path.isdir(brand_dir) or car_brand not in LABEL:
            continue
        print(f"Processing brand: {car_brand}")
        for file_name in os.listdir(brand_dir):
            file_path = os.path.join(brand_dir, file_name)
            ext = os.path.splitext(file_name)[-1].lower()

            # Check valid extension
            if ext not in VALID_EXTENSIONS:
                continue

            # Validate the image
            try:
                img = Image.open(file_path)
                img.verify()  # Verifies the file format
                data.append(file_path)
                labels.append(LABEL[car_brand])
            except (IOError, SyntaxError) as e:
                print(f"Invalid image {file_name}: {e}")

    return np.array(data), np.array(labels)


In [4]:
data = process_directory(root_dir)
paths, label = data

Processing brand: Honda
Invalid image 22521560-22521614.Honda.41.jpg: cannot identify image file '/content/drive/.shortcut-targets-by-id/1Uj0V9URNHpzSHeXHSB89AoGCjGki8Yra/Public/Honda/22521560-22521614.Honda.41.jpg'
Invalid image 22521560-22521614.Honda.36.jpg: cannot identify image file '/content/drive/.shortcut-targets-by-id/1Uj0V9URNHpzSHeXHSB89AoGCjGki8Yra/Public/Honda/22521560-22521614.Honda.36.jpg'
Invalid image 22521560-22521614.Honda.37.jpg: cannot identify image file '/content/drive/.shortcut-targets-by-id/1Uj0V9URNHpzSHeXHSB89AoGCjGki8Yra/Public/Honda/22521560-22521614.Honda.37.jpg'
Processing brand: Hyundai
Processing brand: KIA
Processing brand: Mazda
Processing brand: Mitsubishi
Invalid image 22521463-22521213-22521259.Mitsubishi.57.jpg: cannot identify image file '/content/drive/.shortcut-targets-by-id/1Uj0V9URNHpzSHeXHSB89AoGCjGki8Yra/Public/Mitsubishi/22521463-22521213-22521259.Mitsubishi.57.jpg'
Invalid image 22520348-22520530-22520837.Mitsubishi.18.jpg: cannot identif

In [5]:
print(len(paths))

37785


In [6]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet import preprocess_input
from tensorflow.keras.preprocessing import image
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from PIL import Image
import matplotlib.image as mpimg
import tensorflow as tf

# Khởi tạo mô hình MobileNet, loại bỏ lớp phân loại cuối cùng
model = MobileNetV2(weights='imagenet', include_top=False, pooling='avg')

# Hàm để rút trích đặc trưng từ một ảnh
def extract_features(img_path, model):
    img = image.load_img(img_path, target_size=(224, 224))
    img_data = image.img_to_array(img)
    img_data = np.expand_dims(img_data, axis=0)
    img_data = preprocess_input(img_data)
    features = model.predict(img_data, verbose=0)
    return features.flatten()

features = []

for i in range(len(paths)):
    if (i % 100 == 0):
        print(i)
    feat = extract_features(paths[i], model)
    if feat is not None:
        features.append(feat)

<ipython-input-6-f912b7c8f06c>:11: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  model = MobileNetV2(weights='imagenet', include_top=False, pooling='avg')


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
0
100
200
300
400
500
600
700
800
900
1000
1100
1200
1300
1400


/usr/local/lib/python3.10/dist-packages/PIL/Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


1500
1600
1700
1800
1900
2000
2100
2200
2300
2400
2500
2600
2700
2800
2900
3000
3100
3200
3300
3400
3500
3600
3700
3800
3900
4000
4100
4200
4300
4400
4500
4600
4700
4800
4900
5000
5100
5200
5300
5400
5500
5600
5700
5800
5900
6000
6100
6200
6300
6400
6500
6600
6700
6800
6900
7000
7100
7200
7300
7400
7500
7600
7700
7800
7900
8000
8100
8200
8300
8400
8500
8600
8700
8800
8900
9000
9100
9200
9300
9400
9500
9600
9700
9800
9900
10000
10100
10200
10300
10400
10500
10600
10700
10800
10900
11000
11100
11200
11300
11400
11500
11600
11700
11800
11900
12000
12100
12200
12300
12400
12500
12600
12700
12800
12900
13000
13100
13200
13300
13400
13500
13600
13700
13800
13900
14000
14100
14200
14300
14400
14500
14600
14700
14800
14900
15000
15100
15200
15300
15400
15500
15600
15700
15800
15900
16000
16100
16200
16300
16400
16500
16600
16700
16800
16900
17000
17100
17200
17300
17400
17500
17600
17700
17800
17900
18000
18100
18200
18300
18400
18500
18600
18700
18800
18900
19000
19100
19200
19300
19400
19500

In [7]:
features = np.array(features)  # Ensure features are in a NumPy array
paths = np.array(paths)  # Convert paths to a NumPy array

# Save to an NPZ file
output_file = "features_and_paths.npz"
np.savez(output_file, paths=paths, features=features)